# **Librerias**

In [1]:
import os, re, unicodedata
from collections import Counter, defaultdict
from joblib import Parallel, delayed
import spacy
import pandas as pd
import Levenshtein
import unicodedata
import numpy as np
from itertools import combinations
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_curve, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_predict, RepeatedStratifiedKFold, cross_val_score
from langdetect import detect, detect_langs, DetectorFactory
DetectorFactory.seed = 0   # langdetect es no determinista sin semilla
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.neighbors import NearestNeighbors
from spacy.lang.es.stop_words import STOP_WORDS as stopwords_es
from sklearn.metrics.pairwise import cosine_similarity

C:\Users\aldys\Desktop\NLP\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Paso 1: **Carga y revisión de libros**

In [2]:
# Catálogo: 8747 libros con slug, title, authors, genres, synopsis y otros metadatos. 
# Algunos registros están contaminados; no se indica cuáles. 
catalogo = pd.read_json("Fuente de datos/catalogo_tp3.jsonl", lines=True)

# Muestra: 500 libros del mismo catálogo con la columna contaminado (1 o 0). 
# En los contaminados, la columna fragmento contiene el texto que se insertó. 
muestra = pd.read_csv("Fuente de datos/muestra_etiquetada.csv", keep_default_na=False)

In [3]:
# Renombramos columnas en español
reemplazo_columnas = {
    "source_url": "url",
    "slug": "slug",
    "title": "titulo",
    "authors": "autores",
    "genres": "generos",
    "seed_genre": "generos_semilla",
    "series": "series",
    "series_number": "numero_serie",
    "cover_url": "url_portada",
    "synopsis": "sinopsis",
    "synopsis_chars" : "cantidad_caracteres_sinopsis",
    "scraped_at": "fecha_scrapeo"}
catalogo.rename(columns=reemplazo_columnas, inplace=True)

# Añadimos una columna con la fecha de scrapeo en formato datetime para poder filtrar por fecha
catalogo["fecha_scrapeo"] = pd.to_datetime(catalogo["fecha_scrapeo"])

# Mostramos las primeras filas del catálogo
catalogo.head()

,url,slug,titulo,autores,generos,generos_semilla,series,numero_serie,url_portada,sinopsis,cantidad_caracteres_sinopsis,fecha_scrapeo
0,https://ww3.lectulandia.com/book/de-la-caverna...,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"[Arqueología, Divulgación, Ensayo]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eudald%20C...,La Humanidad ha entrado en una nueva fase evol...,998,2026-10-17 23:11:56+00:00
1,https://ww3.lectulandia.com/book/quienes-somos...,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"[Arqueología, Biología, Ciencias naturales, En...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/David%20Re...,La extracción de ADN de los huesos de nuestros...,1285,2026-10-17 23:11:58+00:00
2,https://ww3.lectulandia.com/book/teotihuacan/,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"[Arqueología, Arte, Ciencias sociales, Ensayo,...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eduardo%20...,Teotihuacan es la primera gran ciudad del cent...,1070,2026-10-17 23:12:01+00:00
3,https://ww3.lectulandia.com/book/tiro-y-las-co...,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],"[Arqueología, Economía, Ensayo, Historia, Polí...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/M%20Eugeni...,Entre los siglos VIII y VI a. C. los fenicios ...,1875,2026-10-17 23:12:03+00:00
4,https://ww3.lectulandia.com/book/la-maldicion-...,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],"[Arqueología, Ensayo, Psicología, Sociología]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Emiliano%2...,Las raíces evolutivas del sufrimiento humano\n...,1613,2026-10-17 23:12:05+00:00


# EDA (Analisis Exploratorio de Datos)

In [4]:
# Eliminamos registros sin sinopsis y duplicados por identificador (slug).
n_inicial = len(catalogo)
catalogo.dropna(subset=["sinopsis"], inplace=True)
n_sin_sinopsis = n_inicial - len(catalogo)

n_previo = len(catalogo)
catalogo.drop_duplicates(subset=["slug"], inplace=True)
n_dup_slug = n_previo - len(catalogo)
print(f"Registros iniciales: {n_inicial} | sin sinopsis: {n_sin_sinopsis} | slug duplicado: {n_dup_slug}")

# Vemos ahora informacion sobre el catalogo
catalogo.info()

Registros iniciales: 8747 | sin sinopsis: 1 | slug duplicado: 0
<class 'pandas.DataFrame'>
Index: 8746 entries, 0 to 8746
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype              
---  ------                        --------------  -----              
 0   url                           8746 non-null   str                
 1   slug                          8746 non-null   str                
 2   titulo                        8746 non-null   str                
 3   autores                       8746 non-null   object             
 4   generos                       8746 non-null   object             
 5   generos_semilla               8746 non-null   str                
 6   series                        2316 non-null   str                
 7   numero_serie                  2298 non-null   float64            
 8   url_portada                   8746 non-null   str                
 9   sinopsis                      8746 non-null   str          

In [5]:
# Dimensiones de nuestro df:
catalogo.shape

(8746, 12)

## Limpieza del texto -> normalización

In [6]:
# Identificamos registros con saltos de línea en la sinopsis
catalogo[catalogo["sinopsis"].str.contains("\n", regex=True, na=False)]["sinopsis"]

0       La Humanidad ha entrado en una nueva fase evol...
1       La extracción de ADN de los huesos de nuestros...
2       Teotihuacan es la primera gran ciudad del cent...
3       Entre los siglos VIII y VI a. C. los fenicios ...
4       Las raíces evolutivas del sufrimiento humano\n...
                              ...                        
8733    El agente Gabriel Syme es reclutado por Scotla...
8734    En esta magnífica obra se cuenta la historia d...
8739    "El bolígrafo de gel verde es el retrato detal...
8743    Cuando una vieja escritora acostumbrada a ment...
8745    ¿Qué futuro tiene un país como España donde la...
Name: sinopsis, Length: 5247, dtype: str

In [7]:
# Reemplazamos cada secuencia de espacios en blanco (espacios, saltos de línea, tabulaciones) por un solo espacio.
# No tocamos ningún otro carácter: algunos van a ser evidencia en el paso 2.
catalogo["sinopsis"] = catalogo["sinopsis"].map(lambda t: re.sub(r"\s+", " ", t).strip())

## **Duplicados**

In [8]:
# Calculamos cuantos duplicados hay en el catalogo (sin contar el original)
n_dup_sinopsis = catalogo.duplicated(subset=['sinopsis']).sum()
print(f"Duplicados antes de eliminar: {n_dup_sinopsis}")

# 1. Detectar los duplicados
duplicados = catalogo[catalogo.duplicated(subset=['sinopsis'], keep=False)]

# 2. Mostrar los casos duplicados
print("Casos duplicados:")
print(duplicados[['titulo', 'sinopsis']])

# 3. Conservar un libro por sinopsis
catalogo = catalogo.drop_duplicates(subset=['sinopsis'])

Duplicados antes de eliminar: 98
Casos duplicados:
                                                 titulo  \
192                                 Historia del tiempo   
605   Introducción a la ciencia II. Ciencias biológicas   
615   Historia del tiempo: Del Big Bang a los agujer...   
736   Comentarios psicológicos sobre las enseñanzas ...   
837                                      Viaje al oeste   
...                                                 ...   
8685                                 Alemán – Unidad 43   
8686                                 Alemán – Unidad 42   
8687                                 Alemán – Unidad 41   
8688                                 Alemán – Unidad 16   
8689                                 Alemán – Unidad 17   

                                               sinopsis  
192   Stephen W. Hawking, físico británico (Oxford, ...  
605   Nueva Guía Asimov de la ciencia es un libro pu...  
615   Stephen W. Hawking, físico británico (Oxford, ...  
736   El

In [9]:
# Las nuevas dimensiones de nuestro df son:
catalogo.shape

(8648, 12)

Pasamos de 8746 registros a 8648. Se eliminaron los 98 repetidos.

### **Versiones casi iguales de una obra**
Comparamos solo libros del **mismo autor** (no todos contra todos). Dos libros se consideran la misma obra si:
1. los títulos normalizados (minúsculas, sin tildes ni signos) tienen un ratio de Levenshtein ≥ 0.85,
2. coincide el número de tomo, volumen o año que aparece en el título (romanos o arábigos),
3. un título contiene al otro (subtítulo, artículo) o son casi idénticos (ratio ≥ 0.95), y
4. las sinopsis tienen un ratio de Levenshtein ≥ 0.90.

La semejanza de cadenas mide la forma y no el significado: «Teatro crítico universal I» y «VII» se parecen mucho y son libros distintos.

In [10]:
# Versiones casi iguales de una obra: mismo autor + título muy parecido + sinopsis muy parecida.
# La semejanza de cadenas mide la forma, no el significado ("Conversaciones con Dios I" vs "II"),
# por eso el criterio combina título, número de tomo/volumen/año y sinopsis.
UMBRAL_TITULO, UMBRAL_SINOPSIS = 0.85, 0.90
PALABRAS_VACIAS_TITULO = {"el", "la", "los", "las", "un", "una", "de", "del", "y", "e", "en"}
ROMANO_O_ARABIGO = re.compile(r"^(?:x{0,3}(?:ix|iv|v?i{0,3})|\d+)$")

def normalizar_titulo(titulo):
    """Minúsculas, sin tildes ni signos de puntuación."""
    t = unicodedata.normalize("NFKD", titulo.lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", t)).strip()

def marca_de_tomo(titulo_norm):
    """Números romanos o arábigos del título: indican tomo, volumen o año."""
    return tuple(p for p in titulo_norm.split() if p and ROMANO_O_ARABIGO.match(p))

def comparar_obras(titulo_a, sinopsis_a, titulo_b, sinopsis_b, usar_sinopsis=True):
    """Devuelve (misma_obra, ratio_titulo, ratio_sinopsis, motivo)."""
    ta, tb = normalizar_titulo(titulo_a), normalizar_titulo(titulo_b)
    r_tit = Levenshtein.ratio(ta, tb)
    if r_tit < UMBRAL_TITULO:
        return False, r_tit, np.nan, "títulos distintos"
    r_sin = Levenshtein.ratio(sinopsis_a, sinopsis_b) if usar_sinopsis else np.nan
    if marca_de_tomo(ta) != marca_de_tomo(tb):
        return False, r_tit, r_sin, "distinto tomo / volumen / año"
    pa, pb = set(ta.split()) - PALABRAS_VACIAS_TITULO, set(tb.split()) - PALABRAS_VACIAS_TITULO
    if not (r_tit >= 0.95 or pa <= pb or pb <= pa):   # una diferencia de contenido ("occidental" vs "oriental")
        return False, r_tit, r_sin, "difieren en palabras de contenido"
    if usar_sinopsis and r_sin < UMBRAL_SINOPSIS:
        return False, r_tit, r_sin, "sinopsis distintas"
    return True, r_tit, r_sin, "mismo título (o subtítulo) y sinopsis"

def encontrar_versiones(df):
    """Compara solo libros del mismo autor. Devuelve un DataFrame con los pares de título parecido."""
    filas, vistos = [], set()
    for autor, grupo in df.explode("autores").groupby("autores"):
        if len(grupo) < 2:
            continue
        for (i, a), (j, b) in combinations(grupo.iterrows(), 2):
            par = (min(i, j), max(i, j))
            if par in vistos:
                continue
            vistos.add(par)
            misma, r_tit, r_sin, motivo = comparar_obras(a["titulo"], a["sinopsis"], b["titulo"], b["sinopsis"])
            if r_tit >= UMBRAL_TITULO:
                filas.append({"indice_1": par[0], "indice_2": par[1], "autor": autor,
                              "titulo_1": df.at[par[0], "titulo"], "titulo_2": df.at[par[1], "titulo"],
                              "ratio_titulo": r_tit, "ratio_sinopsis": r_sin,
                              "misma_obra": misma, "decision": motivo})
    return pd.DataFrame(filas)

versiones = encontrar_versiones(catalogo)
print(f"Pares con título parecido (mismo autor): {len(versiones)} | unidos: {versiones['misma_obra'].sum()}")

cols = ["autor", "titulo_1", "titulo_2", "ratio_titulo", "ratio_sinopsis", "decision"]
print("\nDos pares que decidimos UNIR:")
display(versiones[versiones["misma_obra"]].sort_values("ratio_sinopsis", ascending=False).head(2)[cols])
print("Dos pares que decidimos SEPARAR (sinopsis casi igual, pero tomo/volumen distinto):")
display(versiones[~versiones["misma_obra"]].sort_values("ratio_sinopsis", ascending=False).head(2)[cols])

# Agrupamos las obras unidas (union-find) y conservamos, de cada grupo, la versión con la sinopsis más larga
padre = {}
def raiz(x):
    padre.setdefault(x, x)
    while padre[x] != x:
        padre[x] = padre[padre[x]]
        x = padre[x]
    return x
for i, j in versiones.loc[versiones["misma_obra"], ["indice_1", "indice_2"]].itertuples(index=False):
    padre[raiz(i)] = raiz(j)
grupos = defaultdict(list)
for x in list(padre):
    grupos[raiz(x)].append(x)
a_eliminar = []
for miembros in grupos.values():
    conservar = max(miembros, key=lambda k: len(catalogo.at[k, "sinopsis"]))
    a_eliminar += [k for k in miembros if k != conservar]

n_versiones = len(a_eliminar)
catalogo = catalogo.drop(index=a_eliminar)

print(f"\nResumen del paso 1: {n_inicial} registros iniciales")
print(f"  - sin sinopsis:            {n_sin_sinopsis}")
print(f"  - slug duplicado:          {n_dup_slug}")
print(f"  - sinopsis duplicada:      {n_dup_sinopsis}")
print(f"  - versiones de una obra:   {n_versiones}")
print(f"  = libros disponibles:      {len(catalogo)}")

Pares con título parecido (mismo autor): 549 | unidos: 5

Dos pares que decidimos UNIR:


,autor,titulo_1,titulo_2,ratio_titulo,ratio_sinopsis,decision
203,Maurice Nicoll,Comentarios psicológicos sobre las enseñanzas ...,Comentarios psicológicos sobre las enseñanzas ...,1.000000,0.996337,mismo título (o subtítulo) y sinopsis
199,Mark Twain,El príncipe y el mendigo,Príncipe y mendigo,0.857143,0.993080,mismo título (o subtítulo) y sinopsis


Dos pares que decidimos SEPARAR (sinopsis casi igual, pero tomo/volumen distinto):


,autor,titulo_1,titulo_2,ratio_titulo,ratio_sinopsis,decision
18,Benito Jerónimo Feijoo,"Cartas eruditas, y curiosas II","Cartas eruditas, y curiosas V",0.947368,0.999324,distinto tomo / volumen / año
19,Benito Jerónimo Feijoo,Teatro crítico universal VII,Teatro crítico universal I,0.962963,0.999297,distinto tomo / volumen / año



Resumen del paso 1: 8747 registros iniciales
  - sin sinopsis:            1
  - slug duplicado:          0
  - sinopsis duplicada:      98
  - versiones de una obra:   5
  = libros disponibles:      8643


**Decisiones tomadas** (549 pares de mismo autor con título parecido):
- **5 pares unidos**: títulos idénticos o con artículo distinto y sinopsis casi iguales, p. ej. «El príncipe y el mendigo» / «Príncipe y mendigo» (sinopsis 0.99) y los dos «Comentarios psicológicos sobre las enseñanzas de Gurdjieff y Ouspensky» (0.996). De cada grupo se conserva la versión con la sinopsis más larga.
- **544 pares separados**, sobre todo por tener distinto tomo, volumen o año: «Teatro crítico universal I» / «VII», «Cartas eruditas, y curiosas II» / «V», «La Perla número 5» / «8», «Patente de corso, 2013» / «2015». Aunque su sinopsis sea casi idéntica (0.99), son libros distintos y se conservan.

**Resumen del paso 1** (de 8747 registros a 8643 libros disponibles):
- sin sinopsis: 1
- identificador (`slug`) duplicado: 0
- sinopsis duplicada exacta (cursos y series que repiten el texto en todos los volúmenes): 98
- versiones de una misma obra: 5

Lo comprobamos con `shape`:

In [11]:
catalogo.shape

(8643, 12)

E imprimimos 5 registros del catalogo filtrado:

In [12]:
catalogo.head(5)

,url,slug,titulo,autores,generos,generos_semilla,series,numero_serie,url_portada,sinopsis,cantidad_caracteres_sinopsis,fecha_scrapeo
0,https://ww3.lectulandia.com/book/de-la-caverna...,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"[Arqueología, Divulgación, Ensayo]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eudald%20C...,La Humanidad ha entrado en una nueva fase evol...,998,2026-10-17 23:11:56+00:00
1,https://ww3.lectulandia.com/book/quienes-somos...,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"[Arqueología, Biología, Ciencias naturales, En...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/David%20Re...,La extracción de ADN de los huesos de nuestros...,1285,2026-10-17 23:11:58+00:00
2,https://ww3.lectulandia.com/book/teotihuacan/,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"[Arqueología, Arte, Ciencias sociales, Ensayo,...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eduardo%20...,Teotihuacan es la primera gran ciudad del cent...,1070,2026-10-17 23:12:01+00:00
3,https://ww3.lectulandia.com/book/tiro-y-las-co...,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],"[Arqueología, Economía, Ensayo, Historia, Polí...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/M%20Eugeni...,Entre los siglos VIII y VI a. C. los fenicios ...,1875,2026-10-17 23:12:03+00:00
4,https://ww3.lectulandia.com/book/la-maldicion-...,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],"[Arqueología, Ensayo, Psicología, Sociología]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Emiliano%2...,Las raíces evolutivas del sufrimiento humano U...,1613,2026-10-17 23:12:05+00:00


# Paso 2: **Auditoría del catálogo**

## **División de oraciones y medición**

In [13]:
def oraciones(texto):
    partes = re.split(r"(?<=[.!?])\s+|\n", texto)
    return [p.strip() for p in partes if len(p.strip()) >= 15]

In [14]:
def medir(nombre, marcado, datos, columna = "contaminado"):
    y = datos[columna]
    p = precision_score(y, marcado, zero_division=0)
    r = recall_score(y, marcado)
    print(f"{nombre}: precisión {p:.2f}, recall {r:.2f}")

## **División del dataset**

In [15]:
desarrollo, prueba = train_test_split( 
    muestra, 
    test_size=0.3, 
    random_state=42, 
    stratify=muestra["contaminado"]) 

print(len(desarrollo), len(prueba)) 
# Resultado esperado: 350 150 

350 150


## **Expresiones regulares**

In [16]:
# ---------- 2.1 Regex ----------
TLD = r"(?:com|net|org|io|info|ar|es|co|app|example|test|invalid|local)"
patrones = {
    "url": re.compile(rf"(?:https?://|www\.)\S+|\b[a-z0-9-]+\.{TLD}\b(?:/\S*)?", re.I),
    "correo": re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b"),
    "telefono": re.compile(r"(?<!\w)(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?)\d{3,4}[\s.-]?\d{4}(?!\w)"),
}

In [17]:
def regex_contaminado(texto):
    return int(any(p.search(texto) for p in patrones.values()))

In [18]:
# Revisión de los errores de la regex sobre la muestra de desarrollo
d = desarrollo.copy()
d["pred"] = d["synopsis"].map(regex_contaminado)

print("Falsos positivos (sinopsis legítimas que la regex marca):")
for _, f in d[(d.pred == 1) & (d.contaminado == 0)].iterrows():
    for nombre, p in patrones.items():
        m = p.search(f["synopsis"])
        if m:
            print(f"  [{nombre}] {f['slug']}: {m.group()!r}")

print("\nFalsos negativos (contaminados que la regex no ve), ejemplos:")
for fr in d[(d.pred == 0) & (d.contaminado == 1)]["fragmento"].head(8):
    print("  -", ascii(fr))

Falsos positivos (sinopsis legítimas que la regex marca):
  [url] diverso-es: 'Diverso.es'
  [url] que-pasaria-si: 'http://www.xkcd.com'

Falsos negativos (contaminados que la regex no ve), ejemplos:
  - 'El libro m\xe1s esperado del a\xf1o llega por fin en edici\xf3n digital completa. Le\xe9 este y miles de libros m\xe1s sin publicidad. Activ\xe1 tu prueba gratuita desde tu celular.'
  - 'Consegu\xed este t\xedtulo y m\xe1s de 50.000 libros en la app LetraTotal: el primer mes es sin cargo.'
  - 'Si te gust\xf3 esta rese\xf1a, compartila con tus amigos y particip\xe1 del sorteo de un lector electr\xf3nico.'
  - 'Entr\xe1 a libr\u043eexpress.invalid y le\xe9 sin l\xedmites.'
  - 'Gan\xe1 dinero desde tu casa recomendando libros. Cupos limitados, consult\xe1 requisitos.'
  - 'Le\xe9 este y miles de libros m\xe1s sin publicidad. Activ\xe1 tu prueba gratuita desde tu celular.'
  - 'Si te gust\xf3 esta rese\xf1a, compartila con tus amigos y particip\xe1 del sorteo de un lector electr\xf3nic

## **Normalización**

In [19]:
HOMOGLIFOS = str.maketrans({
    "а":"a","е":"e","о":"o","р":"p","с":"c","у":"y","х":"x","і":"i","ј":"j","ѕ":"s","ԁ":"d",
    "ο":"o","ν":"v","ɑ":"a","А":"A","Е":"E","О":"O","Р":"P","С":"C","Х":"X","І":"I"})

def normalizar(texto):
    t = unicodedata.normalize("NFKC", texto)
    # 1) invisibles (categoría Cf: ancho cero, word joiner, guion blando)
    t = "".join(c for c in t if unicodedata.category(c) != "Cf")
    # 2) letras de otro alfabeto que parecen latinas
    t = t.translate(HOMOGLIFOS)
    # 3) signos escritos como palabras o entre corchetes
    t = re.sub(r"\s*[\[\(]\s*(?:\.|punto|dot)\s*[\]\)]\s*", ".", t, flags=re.I)
    t = re.sub(r"\s+(?:punto|dot)\s+(?=\w)", ".", t, flags=re.I)
    t = re.sub(r"\s+(?:barra|slash)\s+", "/", t, flags=re.I)
    t = re.sub(r"\s*[\[\(]\s*(?:arroba|at)\s*[\]\)]\s*|\s+arroba\s+", "@", t, flags=re.I)
    # 4) letras separadas por espacios: "h t t p s : / /" -> "https://"
    t = re.sub(r"(?<!\S)(?:\S ){2,}\S(?!\S)", lambda mo: mo.group().replace(" ", ""), t)
    t = re.sub(r"(https?)\s*:\s*/\s*/\s*", r"\1://", t, flags=re.I)
    t = re.sub(r"\bwww\s*\.\s*", "www.", t, flags=re.I)
    # pegar el TLD solo si es inequívoco (no "es"/"co": rompería ". Es" de inicio de oración)
    t = re.sub(r"\s*\.\s*(?=(?:example|test|invalid|com|net|org)\b)", ".", t)
    return re.sub(r"\s+", " ", t).strip()

In [20]:
# ---------- Medición ----------
for nombre, df in [("desarrollo", desarrollo), ("prueba", prueba)]:
    medir(f"{nombre} | regex", df["synopsis"].map(regex_contaminado), df)
    medir(f"{nombre} | normalizar + regex", df["synopsis"].map(lambda t: regex_contaminado(normalizar(t))), df)

# Dos casos recuperados por la normalización (desarrollo)
d = desarrollo.copy()
d["cruda"] = d["synopsis"].map(regex_contaminado)
d["norm"] = d["synopsis"].map(lambda t: regex_contaminado(normalizar(t)))
for f in d[(d.cruda == 0) & (d.norm == 1)]["fragmento"].head(2):
    print(ascii(f), "->", normalizar(f))


desarrollo | regex: precisión 0.94, recall 0.41


desarrollo | normalizar + regex: precisión 0.95, recall 0.60
prueba | regex: precisión 1.00, recall 0.40


prueba | normalizar + regex: precisión 1.00, recall 0.63


'Entr\xe1 a libr\u043eexpress.invalid y le\xe9 sin l\xedmites.' -> Entrá a libroexpress.invalid y leé sin límites.
'Link en nuestro sitio: h t t p s : / / epubhub [.] example' -> Link en nuestro sitio: https://epubhub.example


## **Detección de idiomas**

In [21]:
# --- A) Idioma de la sinopsis completa ---
def idioma_completo(texto):
    try:
        return detect(texto)
    except Exception:
        return "?"

d = desarrollo.copy()
d["idioma"] = d["synopsis"].map(idioma_completo)
print(pd.crosstab(d["contaminado"], d["idioma"] == "es"))
medir("sinopsis completa != es", (d["idioma"] != "es").astype(int), d)

idioma       True
contaminado      
0             280
1              70
sinopsis completa != es: precisión 0.00, recall 0.00


In [22]:
# --- B) Idioma por oración ---
def prob_no_es(texto):
    """(idioma, probabilidad) del idioma no español más probable, o None."""
    try:
        langs = detect_langs(texto)
    except Exception:
        return None
    for l in langs:
        if l.lang != "es":
            return l.lang, l.prob
    return None

IDIOMAS_EXTRA = {"en", "pt"}
MIN_PALABRAS, UMBRAL = 8, 0.90

def idioma_extranjero(texto):
    for o in oraciones(texto):
        if len(o.split()) < MIN_PALABRAS or o.isupper():
            continue
        x = prob_no_es(o)
        if x and x[0] in IDIOMAS_EXTRA and x[1] >= UMBRAL:
            return True
    return False

for nombre, df in [("desarrollo", desarrollo), ("prueba", prueba)]:
    medir(f"{nombre} | idioma", df["synopsis"].map(idioma_extranjero).astype(int), df)


desarrollo | idioma: precisión 1.00, recall 0.13


prueba | idioma: precisión 0.89, recall 0.27


In [23]:
# Justificación de MIN_PALABRAS y UMBRAL con la muestra de desarrollo:
# calculamos una sola vez (palabras, idioma no español más probable) de cada oración y barremos ambos valores.
def detalle_oraciones(texto):
    return [(len(o.split()), prob_no_es(o)) for o in oraciones(texto) if not o.isupper()]

detalle_dev = desarrollo["synopsis"].map(detalle_oraciones)

def marcar(detalle, min_palabras, umbral):
    return int(any(n >= min_palabras and x and x[0] in IDIOMAS_EXTRA and x[1] >= umbral for n, x in detalle))

filas = []
for mp in [3, 5, 8, 10, 12]:
    for um in [0.5, 0.7, 0.9, 0.99]:
        marca = detalle_dev.map(lambda d_: marcar(d_, mp, um))
        filas.append({"min_palabras": mp, "umbral": um,
                      "precisión": precision_score(desarrollo["contaminado"], marca, zero_division=0),
                      "recall": recall_score(desarrollo["contaminado"], marca),
                      "falsos positivos": int(((marca == 1) & (desarrollo["contaminado"] == 0)).sum())})
display(pd.DataFrame(filas).round(2))

,min_palabras,umbral,precisión,recall,falsos positivos
0,3,0.50,0.43,0.19,17
1,3,0.70,0.44,0.17,15
2,3,0.90,0.50,0.17,12
3,3,0.99,0.50,0.17,12
4,5,0.50,0.59,0.14,7
5,5,0.70,0.62,0.14,6
6,5,0.90,0.71,0.14,4
7,5,0.99,0.71,0.14,4
8,8,0.50,0.82,0.13,2
9,8,0.70,0.82,0.13,2


## **Reconocimiento de entidades**

In [24]:
nlp = spacy.load('es_core_news_lg')

In [25]:
# Oraciones de los fragmentos contaminados (desarrollo) y una cantidad igual de oraciones limpias
oraciones_fragmento = [o for f in desarrollo.loc[desarrollo["contaminado"] == 1, "fragmento"] for o in oraciones(f)]
pool_limpias = [o for s in desarrollo.loc[desarrollo["contaminado"] == 0, "synopsis"] for o in oraciones(s)]
rng_ner = np.random.default_rng(42)
oraciones_limpias = [str(o) for o in rng_ner.choice(pool_limpias, size=len(oraciones_fragmento), replace=False)]
print("Oraciones de fragmentos:", len(oraciones_fragmento), "| oraciones limpias:", len(oraciones_limpias))

Oraciones de fragmentos: 88 | oraciones limpias: 88


In [26]:
def extraer_entidades(lista):
    """Lista de (texto en minúsculas, etiqueta) de cada entidad encontrada en las oraciones."""
    return [(e.text.lower(), e.label_) for doc in nlp.pipe(lista) for e in doc.ents]

ents_cont = extraer_entidades(oraciones_fragmento)
ents_limp = extraer_entidades(oraciones_limpias)

print("Entidades por etiqueta | fragmentos contaminados:", dict(Counter(l for _, l in ents_cont)))
print("Entidades por etiqueta | oraciones limpias:      ", dict(Counter(l for _, l in ents_limp)))

textos_cont, textos_limp = {t for t, _ in ents_cont}, {t for t, _ in ents_limp}
print("\nEntidades solo en el texto insertado:", sorted(textos_cont - textos_limp)[:40])
print("Entidades compartidas:", sorted(textos_cont & textos_limp))

# ¿Cuántas oraciones tienen al menos una entidad ORG / MISC (sitios, marcas)?
def prop_con_etiqueta(lista, etiquetas=("ORG", "MISC")):
    return np.mean([any(e.label_ in etiquetas for e in doc.ents) for doc in nlp.pipe(lista)])
print(f"\nOraciones con ORG/MISC | fragmentos: {prop_con_etiqueta(oraciones_fragmento):.2f} | limpias: {prop_con_etiqueta(oraciones_limpias):.2f}")

Entidades por etiqueta | fragmentos contaminados: {'MISC': 37, 'LOC': 15, 'PER': 16, 'ORG': 2}
Entidades por etiqueta | oraciones limpias:       {'PER': 40, 'MISC': 19, 'LOC': 20, 'ORG': 3}

Entidades solo en el texto insertado: ['baixe', 'capituloplus', 'com', 'compartila', 'descargalo', 'download this ebook for free', 'epub', 'epub sin registro', 'epubmax', 'ganá', 'get unlimited access to thousands of bestsellers with our limited time offer!', 'grátis agora mesmo', 'https://capituloclub.example/no-las-llames-chicas-llamalas-', 'https://capituloplus.example/historias-desde-la-cadena-de-m', 'lectorplus', 'leia', 'letraclub', 'letralibre', 'letramax', 'letratotal', 'librolibre', 'libroya', 'librоexpress.invalid', 'lеctorflаsh.test', 'nossa', 'oferta', 'pdf', 'sem', 'sem cadastro', 'sign-up required', 'tomoclub', 'tomohub', 'ventas@paginatotal.example', 'visitá', 'visitá capitu\u200bloflash.invalid', 'visitá libro\u200blibre.test', 'visitá pagin\u2060azona.invalid', 'visitá rela\u200bto

**Resultados del NER (spaCy `es_core_news_lg`) sobre 88 oraciones de fragmentos contaminados y 88 oraciones limpias de la muestra de desarrollo:**
- Etiquetas en los fragmentos: MISC 37, PER 16, LOC 15, ORG 2. En las oraciones limpias: PER 40, LOC 20, MISC 19, ORG 3.
- **No hay ninguna entidad en común** entre ambos grupos. Las entidades exclusivas del texto insertado son nombres de sitios o marcas inventados («letramax», «libroya», «epubmax»…), URLs completas, formatos («pdf», «epub») y palabras en portugués o inglés («baixe», «leia», «sem»).
- Una oración tiene al menos una entidad ORG/MISC el 30 % de las veces en los fragmentos y el 16 % en las limpias.

**¿Sirve para detectar o solo para explicar?** Sirve para **explicar**. spaCy etiqueta los sitios inventados de forma inconsistente (a veces como MISC, a veces como LOC o PER) y las sinopsis legítimas también tienen muchas entidades MISC (títulos de obras), así que la etiqueta por sí sola no separa bien las dos clases. Resulta útil para mostrar qué tipo de texto agregó la contaminación, pero para detectarla ya funcionan mejor la regex y los clasificadores.

## **Clasificación de oraciones**

In [27]:
oraciones_desarrollo = []
etiquetas_desarrollo = []
libros_desarrollo = []   # NUEVO: índice del libro de cada oración, sirve para agrupar en la validación

for idx, fila in desarrollo.iterrows():

    if pd.notna(fila["fragmento"]) and fila["fragmento"].strip():
        # Las oraciones del fragmento son positivas (texto promocional conocido)
        for oracion in oraciones(fila["fragmento"]):
            oraciones_desarrollo.append(normalizar(oracion))
            etiquetas_desarrollo.append(1)
            libros_desarrollo.append(idx)

    elif fila["contaminado"] == 0:
        # Las oraciones de sinopsis limpias son negativas
        for oracion in oraciones(fila["synopsis"]):
            oraciones_desarrollo.append(normalizar(oracion))
            etiquetas_desarrollo.append(0)
            libros_desarrollo.append(idx)

# Pasamos a arrays para poder indexar con máscaras de numpy
y_oraciones = np.array(etiquetas_desarrollo)
libros_desarrollo = np.array(libros_desarrollo)

# Control: cuántas oraciones hay de cada clase (esperamos muchas más negativas)
print("Oraciones positivas:", (y_oraciones == 1).sum())
print("Oraciones negativas:", (y_oraciones == 0).sum())


Oraciones positivas: 88
Oraciones negativas: 2007


In [28]:
# --- Modelo A: TF-IDF + regresión logística ---
# DECISIÓN: acá NO usamos el TfidfVectorizer por defecto (palabras) sino n-gramas de CARACTERES (2 a 5).
# Justificación: lo que buscamos en este detector son cadenas literales (URLs, dominios inventados como
# "letramax.example", correos, "h t t p s"). Con palabras completas, cada dominio nuevo es un token desconocido
# y no aporta nada; con n-gramas de caracteres, los trozos "http", ".example", ".test", "www." sí se repiten.
# La comparación con el vectorizador por defecto está en la celda siguiente al cálculo de umbrales:
# a igual precisión (0.90) la ventaja medida sobre las oraciones de desarrollo es mínima (recall 0.98 vs 0.97).
# Se mantiene la elección por robustez ante dominios y ofuscaciones que no están en la muestra, pero
# NO es una diferencia significativa: es una decisión de diseño, no un resultado.
# (En el paso 5, que clasifica preferencias, SÍ se usa la configuración por defecto que pide la consigna.)
modelo_tfidf = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True),
    LogisticRegression(class_weight="balanced", max_iter=1000),)

# --- Modelo B: embeddings + regresión logística ---
# Codificador multilingüe (cambialo por el que hayan usado en el paso 5, si es otro).
# Convierte cada oración en un vector que representa su SENTIDO, no sus letras.
codificador = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
X_emb_desarrollo = codificador.encode(oraciones_desarrollo, show_progress_bar=True)

modelo_emb = LogisticRegression(class_weight="balanced", max_iter=1000)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  28%|██▊       | 55/199 [00:00<00:00, 545.15it/s]

Loading weights:  55%|█████▌    | 110/199 [00:00<00:00, 542.09it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 916.23it/s]

Batches:   0%|          | 0/66 [00:00<?, ?it/s]

Batches:   2%|▏         | 1/66 [00:02<02:27,  2.27s/it]

... (61 outputs left out)

Batches:  95%|█████████▌| 63/66 [00:43<00:00,  5.20it/s]

Batches:  97%|█████████▋| 64/66 [00:43<00:00,  5.34it/s]

Batches:  98%|█████████▊| 65/66 [00:43<00:00,  5.48it/s]

Batches: 100%|██████████| 66/66 [00:43<00:00,  1.52it/s]

In [29]:
# Validación cruzada AGRUPADA por libro: las oraciones de un mismo libro nunca quedan
# a la vez en entrenamiento y validación. Sin esto, las probabilidades salen infladas
# y el umbral queda mal elegido.
cv = GroupKFold(n_splits=5)

# Probabilidad "fuera de muestra" de ser positiva para cada oración de desarrollo
prob_tfidf = cross_val_predict(
    modelo_tfidf, oraciones_desarrollo, y_oraciones,
    groups=libros_desarrollo, cv=cv, method="predict_proba")[:, 1]

prob_emb = cross_val_predict(
    modelo_emb, X_emb_desarrollo, y_oraciones,
    groups=libros_desarrollo, cv=cv, method="predict_proba")[:, 1]

def elegir_umbral(y, prob, precision_objetivo=0.90):
    """Devuelve el menor umbral cuya precisión alcanza `precision_objetivo`.
    El menor umbral es el que deja más recall manteniendo esa precisión."""
    precision, recall, umbrales = precision_recall_curve(y, prob)
    # `precision` y `recall` tienen un elemento más que `umbrales`: recortamos el último
    cumplen = np.where(precision[:-1] >= precision_objetivo)[0]
    if len(cumplen) == 0:
        raise ValueError("Ningún umbral alcanza la precisión objetivo")
    i = cumplen[0]   # los umbrales vienen ordenados de menor a mayor
    return umbrales[i], precision[i], recall[i]

umbral_tfidf, p_t, r_t = elegir_umbral(y_oraciones, prob_tfidf)
umbral_emb,   p_e, r_e = elegir_umbral(y_oraciones, prob_emb)

print(f"TF-IDF     | umbral {umbral_tfidf:.3f} | precisión {p_t:.2f} | recall {r_t:.2f} (por oración)")
print(f"Embeddings | umbral {umbral_emb:.3f} | precisión {p_e:.2f} | recall {r_e:.2f} (por oración)")


TF-IDF     | umbral 0.327 | precisión 0.91 | recall 0.98 (por oración)
Embeddings | umbral 0.778 | precisión 0.90 | recall 0.97 (por oración)


In [ ]:
# Comparación a igual precisión (0.90 por oración, validación agrupada por libro):
# TF-IDF por palabras (configuración por defecto) vs TF-IDF por n-gramas de caracteres (el que usamos).
modelo_tfidf_palabras = make_pipeline(TfidfVectorizer(), LogisticRegression(class_weight="balanced", max_iter=1000))
prob_palabras = cross_val_predict(modelo_tfidf_palabras, oraciones_desarrollo, y_oraciones,
                                  groups=libros_desarrollo, cv=cv, method="predict_proba")[:, 1]
try:
    u_p, p_p, r_p = elegir_umbral(y_oraciones, prob_palabras)
    print(f"TF-IDF por palabras     | umbral {u_p:.3f} | precisión {p_p:.2f} | recall {r_p:.2f} (por oración)")
except ValueError:
    print("TF-IDF por palabras     | ningún umbral alcanza precisión 0.90")
print(f"TF-IDF por caracteres   | umbral {umbral_tfidf:.3f} | precisión {p_t:.2f} | recall {r_t:.2f} (por oración)")

TF-IDF por palabras     | umbral 0.431 | precisión 0.90 | recall 0.97 (por oración)
TF-IDF por caracteres   | umbral 0.327 | precisión 0.91 | recall 0.98 (por oración)


In [30]:
# Ahora sí entrenamos cada modelo con TODAS las oraciones de desarrollo
modelo_tfidf.fit(oraciones_desarrollo, y_oraciones)
modelo_emb.fit(X_emb_desarrollo, y_oraciones)

def prob_max_lote(sinopsis, modelo, usa_embeddings, batch_size=64):
    """Probabilidad de la oración más sospechosa de cada sinopsis (array alineado con `sinopsis`).
    Una sinopsis es tan sospechosa como su oración más sospechosa."""
    ors, dueno = [], []
    for k, s in enumerate(sinopsis):
        for o in oraciones(s):
            ors.append(normalizar(o))
            dueno.append(k)
    salida = np.zeros(len(sinopsis))
    if not ors:
        return salida
    X = codificador.encode(ors, batch_size=batch_size, show_progress_bar=False) if usa_embeddings else ors
    np.maximum.at(salida, np.array(dueno), modelo.predict_proba(X)[:, 1])
    return salida

# Probabilidad máxima por sinopsis en el conjunto de prueba
prueba = prueba.copy()
prueba["p_tfidf"] = prob_max_lote(prueba["synopsis"].tolist(), modelo_tfidf, False)
prueba["p_emb"]   = prob_max_lote(prueba["synopsis"].tolist(), modelo_emb, True)

# Sospechosa = alguna oración supera el umbral propio de cada modelo
marca_tfidf = (prueba["p_tfidf"] >= umbral_tfidf).astype(int)
marca_emb   = (prueba["p_emb"]   >= umbral_emb).astype(int)

medir("prueba | TF-IDF",     marca_tfidf, prueba)
medir("prueba | Embeddings", marca_emb,   prueba)

prueba | TF-IDF: precisión 0.91, recall 0.97
prueba | Embeddings: precisión 1.00, recall 0.87


In [31]:
# Libros contaminados de prueba que detecta uno y no el otro
prueba["marca_tfidf"] = marca_tfidf
prueba["marca_emb"] = marca_emb
cont = prueba[prueba["contaminado"] == 1]

print("Solo TF-IDF:", ((cont.marca_tfidf == 1) & (cont.marca_emb == 0)).sum())
print("Solo embeddings:", ((cont.marca_tfidf == 0) & (cont.marca_emb == 1)).sum())
print("Ambos:", ((cont.marca_tfidf == 1) & (cont.marca_emb == 1)).sum())
print(f"Contaminados de prueba detectados: TF-IDF {cont.marca_tfidf.sum()} de {len(cont)} | embeddings {cont.marca_emb.sum()} de {len(cont)}")
print("Ninguno:", ((cont.marca_tfidf == 0) & (cont.marca_emb == 0)).sum())

# Mostramos los fragmentos de cada grupo para ver qué tipo de texto es
for titulo, filtro in [
    ("SOLO TF-IDF (¿URLs y cadenas literales?)", (cont.marca_tfidf == 1) & (cont.marca_emb == 0)),
    ("SOLO EMBEDDINGS (¿frases nuevas con el mismo sentido?)", (cont.marca_tfidf == 0) & (cont.marca_emb == 1)),
    ("NINGUNO (lo que sigue sin detectarse)", (cont.marca_tfidf == 0) & (cont.marca_emb == 0))]:
    print(f"\n--- {titulo} ---")
    for f in cont[filtro]["fragmento"].dropna().head(5):
        print("-", f)


Solo TF-IDF: 3
Solo embeddings: 0
Ambos: 26
Contaminados de prueba detectados: TF-IDF 29 de 30 | embeddings 26 de 30
Ninguno: 1

--- SOLO TF-IDF (¿URLs y cadenas literales?) ---
- Entrá a letrаya.invalid y leé sin límites.
- Link en nuestro sitio: h t t p s : / / lectortotal [.] example
- Descargalo gratis en TomoMax punto example barra cardamomo

--- SOLO EMBEDDINGS (¿frases nuevas con el mismo sentido?) ---

--- NINGUNO (lo que sigue sin detectarse) ---
- Pedí tu copia escribiendo a ventas@tomozona.test


**¿Por qué clasificar oraciones y no la sinopsis completa?**
La contaminación suele ser una sola oración dentro de una sinopsis larga y limpia. Al clasificar el texto entero, la señal se diluye entre muchas palabras normales. Por oración, la frase promocional queda aislada. Además, así se aprovechan los fragmentos como positivos y las oraciones de sinopsis limpias como negativos.

**¿Qué representación detecta mejor los libros contaminados de prueba?** (umbral elegido para una precisión de 0.90 por oración en desarrollo)

| | Contaminados detectados (de 30) | Precisión | Recall | Legítimos marcados |
|---|---|---|---|---|
| TF-IDF (caracteres) | 29 | 0.91 | 0.97 | 3 |
| Embeddings | 26 | 1.00 | 0.87 | 0 |

TF-IDF detecta más contaminados; los embeddings no marcan ningún libro legítimo.

**¿En qué tipos de contaminación gana cada una?**
- **TF-IDF** gana con las cadenas literales y ofuscadas: los 3 libros que detecta solo TF-IDF son «letrаya.invalid» (con una letra cirílica), «h t t p s : / / lectortotal [.] example» y «TomoMax punto example barra cardamomo». Los n-gramas de caracteres reconocen trozos como `.invalid`, `.example` o `http`.
- **Embeddings** no ganan en ningún libro de prueba (0 detectados solo por ellos), pero evitan los 3 falsos positivos de TF-IDF.
- **Ninguno** detecta «Pedí tu copia escribiendo a ventas@tomozona.test»; ese caso lo cubre la regex.

**Generalización al catálogo:** en el catálogo hay mensajes que no están en la muestra. Los embeddings representan el sentido, así que deberían generalizar mejor a frases nuevas con el mismo significado; TF-IDF es mejor con cadenas literales como una URL. En la muestra de prueba esa ventaja de los embeddings no se puede comprobar, porque casi todos los mensajes se repiten con otros nombres de sitio. En el catálogo completo TF-IDF marcó 397 libros y los embeddings 379.

**Sobre la elección de n-gramas de caracteres:** frente al TF-IDF por palabras por defecto, la ventaja a igual precisión es mínima (recall 0.98 vs 0.97 por oración).

## **Encontrar libros copiados**

In [32]:
# 1. Vectorizar las sinopsis del catálogo con TF-IDF
vectorizer_copias = TfidfVectorizer(max_features=10000, stop_words=list(stopwords_es))
X_cat_tfidf = vectorizer_copias.fit_transform(catalogo['sinopsis'])

# 2. Vecino más cercano con métrica coseno (n_neighbors=2: el 1ro es el mismo libro, el 2do el más parecido)
nn_model = NearestNeighbors(n_neighbors=2, metric='cosine', n_jobs=-1).fit(X_cat_tfidf)
distances, indices = nn_model.kneighbors(X_cat_tfidf)

def jaccard(a, b):
    """Similitud de Jaccard sobre conjuntos de palabras."""
    pa, pb = set(re.findall(r"\w+", a.lower())), set(re.findall(r"\w+", b.lower()))
    return len(pa & pb) / len(pa | pb)

# 3. Pares con similitud alta (coseno > 0.75) y SIN autores en común (las series legítimas comparten autor)
UMBRAL_COSENO = 0.75
pares = []
for i in range(len(catalogo)):
    j = indices[i, 1]
    similitud = 1 - distances[i, 1]
    if similitud <= UMBRAL_COSENO:
        continue
    a, b = catalogo.iloc[i], catalogo.iloc[j]
    if set(a['autores']) & set(b['autores']):
        continue
    # La copia es el libro que agrega texto a la sinopsis del otro: el de sinopsis más larga
    copia, original = (a, b) if len(a['sinopsis']) > len(b['sinopsis']) else (b, a)
    pares.append({
        'slug_copia': copia['slug'], 'titulo_copia': copia['titulo'], 'autor_copia': list(copia['autores']),
        'slug_original': original['slug'], 'titulo_original': original['titulo'], 'autor_original': list(original['autores']),
        'similitud': similitud, 'jaccard': jaccard(a['sinopsis'], b['sinopsis'])})

df_pares_copiados = pd.DataFrame(pares).drop_duplicates(subset=['slug_copia', 'slug_original']).reset_index(drop=True)
# Solo se marca la COPIA; el original es un libro legítimo y debe seguir disponible
slugs_copia_detectados = set(df_pares_copiados['slug_copia'])
print(f"Pares sospechosos por copia: {len(df_pares_copiados)} | libros copia marcados: {len(slugs_copia_detectados)}")
df_pares_copiados.head()

Pares sospechosos por copia: 78 | libros copia marcados: 78


,slug_copia,titulo_copia,autor_copia,slug_original,titulo_original,autor_original,similitud,jaccard
0,cronica-del-silencio-2,Crónica del silencio,[Martín Barrionuevo],la-vida-cotidiana-en-pompeya,La vida cotidiana en Pompeya,[Robert Étienne],0.915400,0.853211
1,los-hijos-de-la-llanura-3,Los hijos de la llanura,[Tomás Ledesma],el-nombre-en-el-muro,El nombre en el muro,[Hervé Le Tellier],0.904057,0.877551
2,la-ultima-estacion,La última estación,[Ignacio Barrionuevo],patty-diphusa,Patty Diphusa,[Pedro Almodóvar],0.932177,0.863248
3,memorias-del-bosque-3,Memorias del bosque,[Lucía Vidal],la-profecia-de-orion,La profecía de Orión,[Patrick Geryl],0.955485,0.897260
4,cronica-del-silencio,Crónica del silencio,[Clara Vidal],imperio-victor-conde,Imperio,[Víctor Conde],0.885553,0.847458


## **Combinar y entregar**

In [33]:
# 1. Detectores sobre la parte de prueba de la muestra
def detector_regex(df):
    return df['synopsis'].map(lambda t: regex_contaminado(normalizar(t)))

def detector_idioma(df):
    return df['synopsis'].map(idioma_extranjero).astype(int)

def detector_copias(df):
    return df['slug'].isin(slugs_copia_detectados).astype(int)

pred_prueba = {
    "Regex normalizado": detector_regex(prueba),
    "Idioma extranjero": detector_idioma(prueba),
    "Clasificador TF-IDF": prueba['marca_tfidf'],
    "Clasificador embeddings": prueba['marca_emb'],
    "Libros copiados": detector_copias(prueba)}

# Combinación por OR lógico (un libro es sospechoso si al menos uno lo marca)
combinado = np.zeros(len(prueba), dtype=int)
for v in pred_prueba.values():
    combinado |= v.values.astype(int)
pred_prueba["Combinación (OR)"] = pd.Series(combinado, index=prueba.index)

# 2. Tabla con la precisión y el recall de cada detector y de la combinación
resultados_detectores = []
for nombre, preds in pred_prueba.items():
    resultados_detectores.append({
        "Detector": nombre,
        "Precisión": round(precision_score(prueba['contaminado'], preds, zero_division=0), 2),
        "Recall": round(recall_score(prueba['contaminado'], preds), 2),
        "Libros marcados": int(preds.sum()),
        "Legítimos marcados": int(((preds == 1) & (prueba['contaminado'] == 0)).sum())})
df_eval_detectores = pd.DataFrame(resultados_detectores)
print("Evaluación de detectores en la muestra de prueba:")
display(df_eval_detectores)

Evaluación de detectores en la muestra de prueba:


,Detector,Precisión,Recall,Libros marcados,Legítimos marcados
0,Regex normalizado,1.00,0.63,19,0
1,Idioma extranjero,0.89,0.27,9,1
2,Clasificador TF-IDF,0.91,0.97,32,3
3,Clasificador embeddings,1.00,0.87,26,0
4,Libros copiados,1.00,0.10,3,0
5,Combinación (OR),0.88,1.00,34,4


In [34]:
# 3. Aplicar el sistema al catálogo completo
def _procesar_bloque(textos):
    DetectorFactory.seed = 0   # cada proceso necesita su propia semilla
    return [(regex_contaminado(normalizar(t)), int(idioma_extranjero(t))) for t in textos]

sinopsis_cat = catalogo['sinopsis'].tolist()
bloques = [sinopsis_cat[i:i + 200] for i in range(0, len(sinopsis_cat), 200)]
resultados = Parallel(n_jobs=-1, verbose=5)(delayed(_procesar_bloque)(b) for b in bloques)
preds = [r for bloque in resultados for r in bloque]

catalogo_eval = catalogo.copy()
catalogo_eval['pred_regex'] = [r[0] for r in preds]
catalogo_eval['pred_idioma'] = [r[1] for r in preds]
catalogo_eval['pred_tfidf'] = (prob_max_lote(sinopsis_cat, modelo_tfidf, False) >= umbral_tfidf).astype(int)
catalogo_eval['pred_emb'] = (prob_max_lote(sinopsis_cat, modelo_emb, True) >= umbral_emb).astype(int)
catalogo_eval['pred_copias'] = catalogo_eval['slug'].isin(slugs_copia_detectados).astype(int)

MOTIVOS = {'pred_regex': 'regex', 'pred_idioma': 'idioma_extranjero', 'pred_tfidf': 'clasificador_tfidf',
           'pred_emb': 'clasificador_embeddings', 'pred_copias': 'copia_catalogo'}
print(catalogo_eval[list(MOTIVOS)].sum().rename(index=MOTIVOS))

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.


[Parallel(n_jobs=-1)]: Done  30 out of  44 | elapsed:   56.5s remaining:   26.3s


[Parallel(n_jobs=-1)]: Done  39 out of  44 | elapsed:  1.1min remaining:    8.6s


[Parallel(n_jobs=-1)]: Done  44 out of  44 | elapsed:  1.2min finished


regex                      203
idioma_extranjero          112
clasificador_tfidf         397
clasificador_embeddings    379
copia_catalogo              78
dtype: int64


In [35]:
# 4. Guardar sospechosos.csv (slug, motivo)
motivos_por_slug = defaultdict(list)
for col, nombre in MOTIVOS.items():
    for slug in catalogo_eval.loc[catalogo_eval[col] == 1, 'slug']:
        motivos_por_slug[slug].append(nombre)

# Los contaminados de la muestra etiquetada ya sabemos que lo están: también salen del catálogo limpio
for slug in set(muestra.loc[muestra['contaminado'] == 1, 'slug']) & set(catalogo['slug']):
    motivos_por_slug[slug].append('muestra_etiquetada')

sospechosos_df = pd.DataFrame({"slug": list(motivos_por_slug), "motivo": [", ".join(m) for m in motivos_por_slug.values()]})
sospechosos_df.to_csv("sospechosos.csv", index=False)
print(f"Total de libros sospechosos guardados en 'sospechosos.csv': {len(sospechosos_df)}")

# 5. Quitar esos libros y guardar catalogo_limpio.csv
catalogo_limpio = catalogo[~catalogo['slug'].isin(set(sospechosos_df['slug']))].copy()
catalogo_limpio.to_csv("catalogo_limpio.csv", index=False)
print(f"Catálogo limpio guardado como 'catalogo_limpio.csv'. Registros restantes: {len(catalogo_limpio)}")

Total de libros sospechosos guardados en 'sospechosos.csv': 611


Catálogo limpio guardado como 'catalogo_limpio.csv'. Registros restantes: 8032


# **Paso 3: Registrar las preferencias de un lector**

In [5]:
RUTA_PREFERENCIAS = "preferencias.csv"
MIN_VALIDAS, MIN_POR_CLASE = 100, 20

# ---------------------------------------------------------------------------------------------
# LECTOR SIMULADO (las respuestas NO las da una persona real)
# ---------------------------------------------------------------------------------------------
# Para poder ejecutar y probar todo el notebook, las respuestas se generan al azar con una semilla fija.
# Para usar un lector real: SIMULAR_LECTOR = False y completar la columna `interes` del archivo.
SIMULAR_LECTOR = True

# Por qué 60 % / 30 % / 10 %:
#  - 60 % de 1 (me interesa): un lector con gustos amplios que acepta la mayoría de los libros.
#  - 30 % de 0 (no me interesa): un rechazo claro, menor que la aceptación, pero suficiente para tener ejemplos negativos.
#  - 10 % de vacíos (no sabe): reproduce las abstenciones del enunciado, que deben conservarse como NaN y no contarse como 0.
# Sobre las respuestas válidas (descartando los vacíos) queda una proporción de 2 a 1 en teoría (≈ 67 % de 1 y 33 % de 0); con la semilla fija salieron 62 unos y 38 ceros (62 % y 38 %).
# Consecuencias para el resto del trabajo:
#  - Con 150 libros esperamos ≈ 90 unos, ≈ 45 ceros y ≈ 15 vacíos: se supera con margen el mínimo de 20 por clase,
#    y el recorrido se corta al llegar a 100 respuestas válidas (≈ 110 libros en teoría; con la semilla fija se cortó en 107, con 7 abstenciones).
#  - Al haber más 1 que 0, el desbalance justifica class_weight="balanced" en los clasificadores.
#  - Al azar, elegir un libro cualquiera acierta ≈ 67 % de las veces en teoría (62 % con las respuestas obtenidas y 60 % en el conjunto de prueba del paso 4): ese es el piso con el que hay que comparar
#    la precisión@5 en el paso 7.
#  - Como las respuestas no dependen del contenido de la sinopsis, ningún modelo puede aprender de ellas:
#    las métricas de los pasos 6 y 7 deben quedar cerca del azar y NO sirven para sacar conclusiones.
P_SI, P_NO, P_NO_SABE = 0.60, 0.30, 0.10

# Mezclamos el catálogo limpio una sola vez con random_state=42; el lector recorre ese orden desde el principio
orden_lector = catalogo_limpio.sample(frac=1, random_state=42).reset_index(drop=True)

def crear_plantilla(n=150):
    """Primeros n libros del orden, con `interes` vacío (1 = sí, 0 = no, vacío = no sabe)."""
    plantilla = orden_lector.head(n)[["slug", "titulo", "sinopsis"]].rename(columns={"titulo": "title", "sinopsis": "synopsis"})
    plantilla["interes"] = np.nan
    return plantilla

if SIMULAR_LECTOR:
    plantilla = crear_plantilla()
    rng = np.random.default_rng(42)
    plantilla["interes"] = rng.choice([1, 0, np.nan], size=len(plantilla), p=[P_SI, P_NO, P_NO_SABE])
    plantilla.to_csv(RUTA_PREFERENCIAS, index=False)
elif not os.path.exists(RUTA_PREFERENCIAS):
    crear_plantilla().to_csv(RUTA_PREFERENCIAS, index=False)
    raise RuntimeError(f"Se creó {RUTA_PREFERENCIAS} vacío: el lector de referencia debe completar la columna `interes` y volver a ejecutar.")

preferencias = pd.read_csv(RUTA_PREFERENCIAS)
orden = {s: k for k, s in enumerate(orden_lector["slug"])}
fuera = ~preferencias["slug"].isin(orden)
if fuera.any():
    print(f"Atención: {fuera.sum()} libros del archivo ya no están en el catálogo limpio y se descartan.")
preferencias = preferencias[~fuera].sort_values("slug", key=lambda s: s.map(orden)).reset_index(drop=True)

# Recorremos en orden y cortamos al reunir >= 100 respuestas válidas y >= 20 de cada clase
c1, c0 = (preferencias["interes"] == 1).cumsum(), (preferencias["interes"] == 0).cumsum()
cumple = ((c1 + c0) >= MIN_VALIDAS) & (c1 >= MIN_POR_CLASE) & (c0 >= MIN_POR_CLASE)
if not cumple.any():
    raise RuntimeError("Faltan respuestas: se necesitan >= 100 válidas y >= 20 de cada clase.")
preferencias = preferencias.iloc[: int(np.argmax(cumple.values)) + 1].copy()
preferencias[["slug", "title", "synopsis", "interes"]].to_csv(RUTA_PREFERENCIAS, index=False)

n_val = int(preferencias["interes"].notna().sum())
print(f"Libros recorridos: {len(preferencias)} | abstenciones (interes vacío): {preferencias['interes'].isna().sum()}")
print("Respuestas válidas:")
print(preferencias["interes"].value_counts().rename({1.0: "1 (sí)", 0.0: "0 (no)"}))
print(f"Proporciones sobre lo recorrido -> 1: {(preferencias['interes']==1).mean():.0%} | 0: {(preferencias['interes']==0).mean():.0%} | vacío: {preferencias['interes'].isna().mean():.0%}")

Libros recorridos: 107 | abstenciones (interes vacío): 7
Respuestas válidas:
interes
1 (sí)    62
0 (no)    38
Name: count, dtype: int64
Proporciones sobre lo recorrido -> 1: 58% | 0: 36% | vacío: 7%


In [6]:
# Si el lector encontró un libro contaminado que el paso 2 no detectó, lo deja con `interes` vacío
# y su slug se agrega acá: pasa a sospechosos.csv con motivo "revisión manual".
slugs_revision_manual = []   # ej.: ["un-slug", "otro-slug"]

if slugs_revision_manual:
    sospechosos_df = pd.concat([sospechosos_df, pd.DataFrame({"slug": slugs_revision_manual, "motivo": "revisión manual"})]).drop_duplicates("slug")
    sospechosos_df.to_csv("sospechosos.csv", index=False)
    print(f"sospechosos.csv actualizado: {len(sospechosos_df)} libros")

# **Paso 4:  Separar los libros para aprender y evaluar**

In [7]:
# Trabajar solo con los registros cuyo interes es 0 o 1 (se descartan las abstenciones)
preferencias = pd.read_csv("preferencias.csv").drop(columns=["conjunto"], errors="ignore")
validas = preferencias[preferencias["interes"].isin([0, 1])].copy()
validas["interes"] = validas["interes"].astype(int)

# División 80 % entrenamiento / 20 % prueba, estratificada por interes
entrenamiento, prueba_pref = train_test_split(
    validas,
    test_size=0.2,
    random_state=42,
    stratify=validas["interes"])

# Los libros de prueba deben quedar fuera del entrenamiento
assert set(entrenamiento["slug"]).isdisjoint(prueba_pref["slug"])

# Guardar en preferencias.csv la columna conjunto (vacía para las abstenciones)
preferencias["conjunto"] = pd.Series(index=preferencias.index, dtype="object")  # object: admite texto
preferencias.loc[entrenamiento.index, "conjunto"] = "entrenamiento"
preferencias.loc[prueba_pref.index, "conjunto"] = "prueba"
preferencias.to_csv("preferencias.csv", index=False)

# Cantidad de libros y de respuestas positivas en cada conjunto
resumen = pd.DataFrame({
    "conjunto": ["entrenamiento", "prueba"],
    "libros": [len(entrenamiento), len(prueba_pref)],
    "positivos (interes=1)": [(entrenamiento["interes"] == 1).sum(), (prueba_pref["interes"] == 1).sum()],
    "negativos (interes=0)": [(entrenamiento["interes"] == 0).sum(), (prueba_pref["interes"] == 0).sum()]})
resumen["% positivos"] = (resumen["positivos (interes=1)"] / resumen["libros"] * 100).round(1)
display(resumen)


,conjunto,libros,positivos (interes=1),negativos (interes=0),% positivos
0,entrenamiento,80,50,30,62.5
1,prueba,20,12,8,60.0


# **Paso 5:  Entrenar dos clasificadores**

In [8]:
X_train_text = entrenamiento["synopsis"]
y_train = entrenamiento["interes"]
X_test_text = prueba_pref["synopsis"]
y_test = prueba_pref["interes"]

# --- Modelo A con TF-IDF ---
vectorizer_tfidf = TfidfVectorizer() # Configuración por defecto
X_train_tfidf = vectorizer_tfidf.fit_transform(X_train_text)
X_test_tfidf = vectorizer_tfidf.transform(X_test_text)

clf_a = LogisticRegression(C=1, max_iter=1000, class_weight="balanced", random_state=42)
clf_a.fit(X_train_tfidf, y_train)
preds_a = clf_a.predict(X_test_tfidf)

print(f"Tamaño matriz TF-IDF entrenamiento: {X_train_tfidf.shape}")
print(f"Tamaño matriz TF-IDF prueba: {X_test_tfidf.shape}")

# --- Modelo B con Embeddings ---
# Usamos el modelo multilingüe de oraciones (ej. paraphrase-multilingual-MiniLM-L12-v2)
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

X_train_emb = embedder.encode(X_train_text.tolist(), show_progress_bar=False)
X_test_emb = embedder.encode(X_test_text.tolist(), show_progress_bar=False)

clf_b = LogisticRegression(C=1, max_iter=1000, class_weight="balanced", random_state=42)
clf_b.fit(X_train_emb, y_train)
preds_b = clf_b.predict(X_test_emb)

print(f"Tamaño matrices Embeddings - Entrenamiento: {X_train_emb.shape}, Prueba: {X_test_emb.shape}")

Tamaño matriz TF-IDF entrenamiento: (80, 3976)
Tamaño matriz TF-IDF prueba: (20, 3976)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  16%|█▌        | 32/199 [00:00<00:00, 277.59it/s]

Loading weights:  30%|███       | 60/199 [00:00<00:00, 215.72it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 642.71it/s]

Tamaño matrices Embeddings - Entrenamiento: (80, 384), Prueba: (20, 384)


# **Paso 6:  Comprobar las predicciones**

In [9]:
print("--- Evaluación Modelo A (TF-IDF) ---")
print(classification_report(y_test, preds_a, zero_division=0))
print("Matriz de Confusión:\n", confusion_matrix(y_test, preds_a))

print("\n--- Evaluación Modelo B (Embeddings) ---")
print(classification_report(y_test, preds_b, zero_division=0))
print("Matriz de Confusión:\n", confusion_matrix(y_test, preds_b))

# Validación cruzada repetida sobre entrenamiento
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

# Pipeline para Modelo A (TF-IDF se ajusta dentro de cada partición)
pipeline_a = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(C=1, max_iter=1000, class_weight="balanced", random_state=42))])
scores_a = cross_val_score(pipeline_a, X_train_text, y_train, cv=rskf, scoring='f1')

# Validación cruzada para Modelo B
scores_b = cross_val_score(clf_b, X_train_emb, y_train, cv=rskf, scoring='f1')

print(f"\nModelo A (TF-IDF) - F1 Clase 1 (CV): Media = {scores_a.mean():.2f}, Desvío = {scores_a.std():.2f}")
print(f"Modelo B (Embeddings) - F1 Clase 1 (CV): Media = {scores_b.mean():.2f}, Desvío = {scores_b.std():.2f}")

--- Evaluación Modelo A (TF-IDF) ---
              precision    recall  f1-score   support

           0       0.45      0.62      0.53         8
           1       0.67      0.50      0.57        12

    accuracy                           0.55        20
   macro avg       0.56      0.56      0.55        20
weighted avg       0.58      0.55      0.55        20

Matriz de Confusión:
 [[5 3]
 [6 6]]

--- Evaluación Modelo B (Embeddings) ---
              precision    recall  f1-score   support

           0       0.40      0.25      0.31         8
           1       0.60      0.75      0.67        12

    accuracy                           0.55        20
   macro avg       0.50      0.50      0.49        20
weighted avg       0.52      0.55      0.52        20

Matriz de Confusión:
 [[2 6]
 [3 9]]



Modelo A (TF-IDF) - F1 Clase 1 (CV): Media = 0.64, Desvío = 0.10
Modelo B (Embeddings) - F1 Clase 1 (CV): Media = 0.63, Desvío = 0.11


In [10]:
# Tabla resumen de la clase 1 para cada modelo, con aciertos y errores
filas = []
for nombre, pred, cv in [("A: TF-IDF", preds_a, scores_a), ("B: Embeddings", preds_b, scores_b)]:
    filas.append({
        "Modelo": nombre,
        "Precisión": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "Aciertos": int((pred == y_test.values).sum()),
        "Errores": int((pred != y_test.values).sum()),
        "F1 CV media": cv.mean(), "F1 CV desvío": cv.std()})
display(pd.DataFrame(filas).round(2))

# Dos libros de prueba (modelo B): un acierto y un error
res = prueba_pref.assign(pred_a=preds_a, pred_b=preds_b)
acierto = res[(res["pred_b"] == res["interes"])].head(1)
error = res[(res["pred_b"] != res["interes"])].head(1)
for etiqueta, df_ in [("ACIERTO", acierto), ("ERROR", error)]:
    for _, r in df_.iterrows():
        tipo = "" if etiqueta == "ACIERTO" else (" (falso positivo)" if r["pred_b"] == 1 else " (falso negativo)")
        print(f"\n[{etiqueta}{tipo}] {r['title']} | real={r['interes']} | A={r['pred_a']} | B={r['pred_b']}")
        print(r["synopsis"][:400], "...")

,Modelo,Precisión,Recall,F1,Aciertos,Errores,F1 CV media,F1 CV desvío
0,A: TF-IDF,0.67,0.50,0.57,11,9,0.64,0.10
1,B: Embeddings,0.60,0.75,0.67,11,9,0.63,0.11



[ACIERTO] El donador de almas | real=1 | A=1 | B=1
Rafael Antiga es un doctor que recibe de su amigo Andrés Esteves un regalo especial, el alma de una persona fallecida. Esta alma es la de Alda, una monja de clausura llamada Sor Teresa que pasa el tiempo en meditaciones, por lo que su alma puede salir cuando quiere, Esteves hace que Alda se aloje en uno de los hemisferios del cerebro del doctor Antiga. La relación al inicio es muy positiva, pero a ...

[ERROR (falso negativo)] Una vida en palabras | real=1 | A=1 | B=0
Una vida en palabras es un diálogo entre Paul Auster y la profesora danesa I. B. Siegumfeldt acerca del oficio, el arte y la vida del escritor. Profundamente documentado, y dando un paso más allá de la biografía, estas páginas están plagadas de sorpresas y revelaciones que nunca han sido compartidas por el escritor, así como enseñanzas que a menudo saltan del campo de la literatura para hablarnos  ...


**Interpretación de los resultados (respuestas simuladas)**

Con 20 libros de prueba (12 positivos) un solo libro mueve la exactitud 5 puntos, así que las métricas son orientativas. Ambos modelos aciertan 11 de 20 (0.55) y se equivocan en 9.

| Modelo | Precisión (1) | Recall (1) | F1 (1) | F1 en validación cruzada |
|---|---|---|---|---|
| A: TF-IDF | 0.67 | 0.50 | 0.57 | 0.64 ± 0.10 |
| B: Embeddings | 0.60 | 0.75 | 0.67 | 0.63 ± 0.11 |

Como referencia, un «modelo» que responda siempre 1 tendría en prueba precisión 0.60, recall 1.00 y F1 0.75, mejor que ambos. Los dos están en el nivel del azar, algo esperable: las respuestas se sortearon sin relación con el contenido de los libros.

- **Acierto — «El donador de almas»** (real = 1; A = 1, B = 1): los dos modelos coinciden con la respuesta del lector, pero como las etiquetas son aleatorias es una coincidencia y no un patrón aprendido.
- **Error — «Una vida en palabras»** (real = 1; A = 1, B = 0): falso negativo del modelo B. La sinopsis describe un diálogo entre Paul Auster y una profesora sobre el oficio de escribir; el modelo le asignó poca probabilidad, pero con etiquetas al azar no hay una causa que corregir.

# **Paso 7:  Convertir las predicciones en recomendaciones**

In [11]:
# Verificamos qué columna de predict_proba corresponde a interes=1
assert list(clf_a.classes_) == [0, 1] and list(clf_b.classes_) == [0, 1]
col_a, col_b = list(clf_a.classes_).index(1), list(clf_b.classes_).index(1)

df_test_eval = prueba_pref.copy()
df_test_eval['prob_a'] = clf_a.predict_proba(X_test_tfidf)[:, col_a]
df_test_eval['prob_b'] = clf_b.predict_proba(X_test_emb)[:, col_b]

# Perfil del lector: promedio de los embeddings de los positivos de ENTRENAMIENTO (nada de prueba)
perfil_lector = X_train_emb[(y_train == 1).values].mean(axis=0).reshape(1, -1)
df_test_eval['sim_perfil'] = cosine_similarity(X_test_emb, perfil_lector).flatten()

listas = {
    "TF-IDF + clasificador": df_test_eval.sort_values('prob_a', ascending=False).head(5),
    "Embeddings + clasificador": df_test_eval.sort_values('prob_b', ascending=False).head(5),
    "Similitud con el perfil": df_test_eval.sort_values('sim_perfil', ascending=False).head(5)}
col_puntaje = {"TF-IDF + clasificador": "prob_a", "Embeddings + clasificador": "prob_b", "Similitud con el perfil": "sim_perfil"}

def precision_at_5(top_df):
    return (top_df['interes'] == 1).sum() / 5

for nombre, top in listas.items():
    print(f"\n=== {nombre} | precision@5 = {precision_at_5(top):.2f} ===")
    display(top[['title', 'interes', col_puntaje[nombre]]].reset_index(drop=True))

print(f"\nPositivos en prueba: {(y_test == 1).sum()} de {len(y_test)}")
print(f"Precisión esperada al elegir al azar: {(y_test == 1).mean():.2f}")


=== TF-IDF + clasificador | precision@5 = 0.60 ===


,title,interes,prob_a
0,Aprendiz de burro,1,0.557546
1,Diccionario de sinónimos y antónimos,0,0.544418
2,El consejero,0,0.536457
3,Historias de familia,1,0.525679
4,El donador de almas,1,0.514917



=== Embeddings + clasificador | precision@5 = 0.40 ===


,title,interes,prob_b
0,Muerte y circunstancia,0,0.921655
1,El consejero,0,0.912499
2,Historias de familia,1,0.879412
3,El donador de almas,1,0.857848
4,Corazón y ciencia,0,0.830188



=== Similitud con el perfil | precision@5 = 0.40 ===


,title,interes,sim_perfil
0,Así paga el diablo,1,0.699679
1,Metafísica 4 en 1 Vol. 1,0,0.679962
2,Viaje al fin de la noche,0,0.640124
3,Crónica de San Gabriel,1,0.633973
4,El día del Watusi,0,0.598276



Positivos en prueba: 12 de 20
Precisión esperada al elegir al azar: 0.60


**¿Los clasificadores recomendaron mejor que la similitud con el perfil?**

| Lista | Aciertos en los primeros 5 | precision@5 |
|---|---|---|
| TF-IDF + clasificador | 3 | 0.60 |
| Embeddings + clasificador | 2 | 0.40 |
| Similitud con el perfil | 2 | 0.40 |
| Azar (proporción de 1 en prueba, 12 de 20) | | 0.60 |

**No se puede afirmar que los clasificadores recomienden mejor.** El clasificador con embeddings empata con la similitud (0.40) y el de TF-IDF le saca un libro de diferencia (3 contra 2 aciertos de 5), pero apenas iguala al azar (0.60). Con 20 libros de prueba esa diferencia de un libro no es significativa. Con respuestas simuladas, ninguna lista debería ser mejor que el azar.

# **Paso 8:  Recomendar libros del catálogo completo**

In [12]:
# Excluir TODOS los libros de preferencias.csv (valorados y abstenciones)
slugs_vistos = set(preferencias['slug'])
candidatos = catalogo_limpio[~catalogo_limpio['slug'].isin(slugs_vistos)].copy()

# Probabilidad de interes=1 con el clasificador B del paso 5, sin volver a ajustarlo
col_pos = list(clf_b.classes_).index(1)
X_candidatos_emb = embedder.encode(candidatos['sinopsis'].tolist(), show_progress_bar=False, batch_size=64)
candidatos['prob_interes'] = clf_b.predict_proba(X_candidatos_emb)[:, col_pos]
candidatos = candidatos.sort_values('prob_interes', ascending=False)

# Revisión: que no sea otra versión de un libro ya visto (mismo autor y misma obra según el criterio del paso 1)
por_autor = defaultdict(list)
for fila in catalogo_limpio[catalogo_limpio['slug'].isin(slugs_vistos)].itertuples():
    for autor in fila.autores:
        por_autor[autor].append((fila.titulo, fila.sinopsis))

def es_version_de_visto(fila):
    return any(comparar_obras(fila.titulo, fila.sinopsis, t, s, usar_sinopsis=False)[0]
               for autor in fila.autores for t, s in por_autor.get(autor, []))

def resumen_extractivo(texto, n_oraciones=2):
    """Las n oraciones más centrales (suma de similitud coseno TF-IDF con las demás), en su orden original."""
    ors = oraciones(texto)
    if len(ors) <= n_oraciones:
        return " ".join(ors)
    try:
        sim = cosine_similarity(TfidfVectorizer().fit_transform(ors))
    except ValueError:   # oraciones sin vocabulario útil
        return " ".join(ors[:n_oraciones])
    return " ".join(ors[i] for i in sorted(np.argsort(sim.sum(axis=1))[-n_oraciones:]))

def recomendar(n=5):
    """Devuelve identificador, título, puntuación y resumen de los n libros más probables no vistos."""
    elegidos = []
    for fila in candidatos.itertuples():
        if es_version_de_visto(fila):
            continue
        elegidos.append({"slug": fila.slug, "titulo": fila.titulo, "puntuacion": fila.prob_interes,
                         "resumen": resumen_extractivo(fila.sinopsis)})
        if len(elegidos) == n:
            break
    return pd.DataFrame(elegidos)

recomendaciones = recomendar(5)
sinopsis_por_slug = candidatos.set_index('slug')['sinopsis']
for r in recomendaciones.itertuples():
    print(f"\n### {r.titulo} (puntuación {r.puntuacion:.3f})")
    print("SINOPSIS:", sinopsis_por_slug[r.slug])
    print("RESUMEN: ", r.resumen)
display(recomendaciones)


### Una sombra de sospecha (puntuación 0.993)
SINOPSIS: Un secreto. Una promesa. Un crimen . Toda familia tiene una oveja negra . La noticia de una tragedia hace añicos la tranquila vida de Gil como profesor de Escritura Creativa en un pequeño pueblo de Vermont. Su hermana y su cuñado han muerto en un violento accidente de tráfico y él se ve obligado a adoptar a su sobrino Matthew, un joven en apariencia encantador y dulce, aunque no parece estar afectado por la muerte de sus padres. Lo que en un inicio es un acto de amor, pronto se convertirá en la peor pesadilla para Gil y su familia: cuando Matthew se inscribe en las clases de escritura de su tío y los relatos que entrega describen brutales escenas donde su nueva familia muere, todo parece salirse de control. Antes de que la ficción supere a la realidad, Gil decide buscar ayuda con la policía, pero ¿qué peligro puede entrañar la imaginación de un adolescente muy talentoso? Hipnótico. Brillante. Un thriller psicológico que explora e

,slug,titulo,puntuacion,resumen
0,una-sombra-de-sospecha,Una sombra de sospecha,0.992733,La noticia de una tragedia hace añicos la tran...
1,el-rabino,El rabino,0.985309,El abuelo le trasmitió su conciencia de perten...
2,arboles-frutales-en-maceta,Árboles frutales en maceta,0.984347,Guía práctica para plantar y cosechar delicios...
3,el-enterrador-de-munecas,El enterrador de muñecas,0.982494,Protegido por el amor de su madre y la amistad...
4,angel-negro,Ángel negro,0.982073,"La hermosa e inteligente Heaven Leigh, hija de..."


**Comentarios del lector sobre las cinco recomendaciones (SIMULADOS, no los escribió una persona)**

Estos comentarios se mantienen separados de las métricas de prueba.

| Libro | ¿Le interesa? | Comentario |
|---|---|---|
| Una sombra de sospecha | Sí | Thriller psicológico con un buen planteo; me engancha. |
| El rabino | Sí | Novela histórica sobre identidad y religión, justo lo que leería. |
| Árboles frutales en maceta | No | Es una guía de jardinería, no me interesa. La sinopsis tiene una sola oración. |
| El enterrador de muñecas | Sí | Suspense en un pueblo chico; me parece atractivo. |
| Ángel negro | No | Melodrama de familias ricas, no es lo mío. |

Aciertan 3 de 5. Una puntuación alta (0.98 o más en las cinco) expresa una estimación del modelo y no garantiza que el libro vaya a gustar, y con respuestas simuladas no tiene significado real.

**Resúmenes extractivos**
- **Funcionó bien — «El enterrador de muñecas»**: las dos oraciones elegidas conservan la premisa (Ben, el pueblo y sus muñecas) y el conflicto (la desaparición de dos chicas y la sospecha sobre Ben).
- **Perdió información — «Ángel negro»**: el resumen se queda con el planteo (la pobreza de Heaven y la mudanza a la mansión) y descarta la última oración, que es la que anuncia el conflicto («un sórdido mundo de relaciones inconfesables»). En «Una sombra de sospecha» también se pierde el cierre que la presenta como un thriller psicológico.
- **«Árboles frutales en maceta»** tiene una sinopsis de una sola oración; no hay nada que resumir.

In [13]:
# --- Perfil del lector con spaCy: frases sustantivas (lema del núcleo) y entidades nombradas ---
def perfil_spacy(sinopsis):
    nucleos, entidades = Counter(), Counter()
    for doc in nlp.pipe(sinopsis.tolist(), batch_size=32):
        for ch in doc.noun_chunks:
            if ch.root.pos_ in ("NOUN", "PROPN") and ch.root.is_alpha and not ch.root.is_stop:
                nucleos[ch.root.lemma_.lower()] += 1
        for e in doc.ents:
            entidades[e.text.lower()] += 1
    return nucleos.most_common(10), entidades.most_common(10)

nuc_pos, ent_pos = perfil_spacy(entrenamiento[entrenamiento['interes'] == 1]['synopsis'])
nuc_neg, ent_neg = perfil_spacy(entrenamiento[entrenamiento['interes'] == 0]['synopsis'])

print("--- Frases sustantivas (núcleo lematizado) ---")
display(pd.DataFrame({"interes=1": nuc_pos, "interes=0": nuc_neg}))
print("--- Entidades nombradas ---")
display(pd.DataFrame({"interes=1": ent_pos, "interes=0": ent_neg}))

--- Frases sustantivas (núcleo lematizado) ---


,interes=1,interes=0
0,"(libro, 29)","(vida, 16)"
1,"(vida, 26)","(historia, 14)"
2,"(mundo, 17)","(mundo, 12)"
3,"(historia, 15)","(obra, 11)"
4,"(año, 12)","(libro, 11)"
5,"(autor, 12)","(año, 10)"
6,"(mujer, 10)","(guerra, 8)"
7,"(hombre, 10)","(siglo, 8)"
8,"(obra, 9)","(especie, 8)"
9,"(poeta, 8)","(extinción, 8)"


--- Entidades nombradas ---


,interes=1,interes=0
0,"(lacan, 4)","(unión soviética, 4)"
1,"(dios, 4)","(bernhard, 3)"
2,"(pdf, 3)","(londres, 3)"
3,"(mario muchnik, 3)","(churchill, 3)"
4,"(parís, 3)","(alemania, 3)"
5,"(houellebecq, 3)","(cáucaso, 3)"
6,"(trudi birger, 3)","(juan ramón jiménez, 2)"
7,"(bello, 3)","(trufo, 2)"
8,"(él, 3)","(hitler, 2)"
9,"(voltaire, 3)","(historia, 2)"


**Comentario del lector sobre el perfil (SIMULADO)**

«La lista de frases sustantivas no describe lo que me gusta: *libro*, *vida*, *historia*, *mundo* y *obra* aparecen en los dos grupos (con más repeticiones en `interes=1`, que tiene más sinopsis: 50 contra 30, por ejemplo *libro* 29 contra 11). Lo único que cambia es que en los que sí me interesan hay *mujer*, *hombre* y *poeta*, y en los que no, *guerra*, *siglo*, *especie* y *extinción*. Las entidades están muy dispersas: casi todas aparecen entre 2 y 4 veces, y no hay un patrón.»

Es el resultado esperado, porque las respuestas se sortearon y el perfil no tiene un gusto real que reflejar. Tampoco hay que sobreinterpretar frecuencias tan bajas. La entidad «pdf» aparece entre las más frecuentes del grupo `interes=1` y suele ser un resto de texto promocional en la sinopsis: conviene revisar si quedó algún libro contaminado sin detectar.